# Imports

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib as mpl

import pmdarima as pm

from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
mpl.rcParams['figure.figsize'] = (14,7)
np.set_printoptions(precision=4)

# Load Data

In [ ]:
df = pd.read_csv("data/multivariate.csv", header=0, parse_dates=True, index_col=0)

In [ ]:
df.index

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
fig, ax1 = plt.subplots()
ax1.plot(df['energy_demand'], c='b', ls = '-', marker = 'o', label=f"Energy demand")
ax1.set_ylabel('Energy Demand (kW)', color='b')
ax1.tick_params(axis='y', labelcolor='b')
ax1.grid(True)
plt.legend(loc='upper left')

ax2 = ax1.twinx()
ax2.plot(df['temperature'], c='r', ls='--', marker = 'x', label=f"Temperatue")
ax2.set_ylabel('Temperature (°C)', color='r')
ax2.tick_params(axis='y', labelcolor='r')
plt.legend(loc='upper right')

plt.title('Energy Demand and Temperature')
plt.tight_layout()
plt.show()

# Split train and test

In [ ]:
n = int(len(df) * 0.8)

In [ ]:
df_train = df[:n]
df_test = df[n:]

In [ ]:
exog_train = df_train['temperature'].to_numpy().reshape(-1,1)
exog_test = df_test['temperature'].to_numpy().reshape(-1,1)

# Auto Arima

In [ ]:
model = pm.auto_arima(
    y=df_train['energy_demand'], X=exog_train,
    start_p=0, d=0, start_q=0,
    max_p=3, max_d=3, max_q=3,
    start_P=0, D=0, start_Q=0,
    max_P=3, max_D=3, max_Q=3,
    seasonal=True, m = 12,
    stepwise=True, # If False, it will iterate for each combination.
    alpha=0.01,
    trace=True)

In [ ]:
model.params()

In [ ]:
model.summary()

# Eval

## Train

In [ ]:
pred_train = model.predict_in_sample(X=exog_train)

In [ ]:
rmse_train = np.sqrt(mean_squared_error(df_train['energy_demand'], pred_train)).item()
rmse_train

In [ ]:
plt.plot(df_train['energy_demand'], label='Ground truth (train)')
plt.plot(pred_train, label = f"In-sample-train predictions")
plt.plot(df_test['energy_demand'][:1], label = f"Train RMSE: {rmse_train:.4f}")
plt.legend()
plt.show()

## Test

In [ ]:
pred_test = model.predict(n_periods=len(df_test), X=exog_test)

In [ ]:
rmse_test = np.sqrt(mean_squared_error(df_test['energy_demand'], pred_test)).item()
rmse_test

In [ ]:
plt.plot(df_test['energy_demand'], label='Ground truth (test)')
plt.plot(pred_test, label = f"In-sample-test predictions")
plt.plot(df_test['energy_demand'][:1], label = f"Test RMSE: {rmse_test:.4f}")
plt.legend()
plt.show()

# Out-of-Sample Predictions

In [ ]:
# fit entire dataset

model.fit(y=df['energy_demand'], X=df['temperature'].to_numpy().reshape(-1,1))

In [ ]:
# periods to predict

pred_periods = 12

In [ ]:
# generate new exogenous variables

gen = np.random.default_rng()

new_exogenous = gen.normal(
    loc=df['temperature'].mean(),
    scale=df['temperature'].std(),
    size=pred_periods)\
        .reshape(-1,1)

In [ ]:
preds = model.predict(n_periods=pred_periods, X=new_exogenous)
preds

In [ ]:
plt.plot(df['energy_demand'][-24:], label=f'Ground truth (last {24} months)')
plt.plot(preds, label = f"Out-of-sample predictions (next {pred_periods} months)")
plt.legend()
plt.show()